# LPPL Analysis for BSE Sensex Crashes
### Multi-start Optimization + Sensitivity Analysis

**For the PLOS ONE Manuscript**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import f
import warnings
import os

warnings.filterwarnings("ignore")

# ====================== SETTINGS ======================
OUTPUT_FOLDER = "./LPPL_Results/"
FILE_PATH = "Crash_Window_1_02_01_1986_to_09_10_1990.csv"  # Change this
CRASH_NAME = "Crash 1"                                     # Change this

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import f
import warnings
import os

warnings.filterwarnings("ignore")

# ====================== USER SETTINGS ======================
OUTPUT_FOLDER = "./LPPL_Results/"                    # Output folder (created automatically)
FILE_PATH = "Crash_Window_1_02_01_1986_to_09_10_1990.csv"  # Change this for each crash
CRASH_NAME = "Crash 1"                              # Change this for each crash

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def lppl(t, A, B, C, tc, beta, omega, phi):
    """Log-Periodic Power Law (LPPL) model"""
    return A + B * (tc - t)**beta * (1 + C * np.cos(omega * np.log(tc - t) + phi))


def lppl_loss(params, t, y):
    """Sum of Squared Errors (SSE)"""
    try:
        return np.sum((y - lppl(t, *params))**2)
    except:
        return 1e10


def fit_single_crash():
    """Main function: Load data, fit LPPL with multi-start, save results and plots"""
    # Load data
    df = pd.read_csv(FILE_PATH)
    df['Close'] = df['Close'].astype(str).str.replace(',', '').astype(float)
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
    df = df.sort_values('Date').reset_index(drop=True)
    
    t = np.array([(date - df['Date'].iloc[0]).days for date in df['Date']])
    y = np.log(df['Close'].values)
    
    # Parameter bounds
    bounds = [(0, None), (None, None), (-0.5, 0.5),
              (t.max(), t.max() + 400), (0.05, 0.95),
              (4.0, 15.0), (0, 2 * np.pi)]
    
    # ====================== MULTI-START OPTIMIZATION ======================
    best_res = None
    best_fun = np.inf
    np.random.seed(42)
    
    print(f"Running multi-start optimization (40 runs) for {CRASH_NAME}...")
    
    for i in range(40):
        init = [float(y.mean()), -0.1, np.random.uniform(-0.4, 0.4),
                float(t.max() + np.random.uniform(15, 80)),
                np.random.uniform(0.1, 0.85),
                np.random.uniform(6, 12),
                np.random.uniform(0, 2 * np.pi)]
        
        res = minimize(lppl_loss, init, args=(t, y), bounds=bounds,
                       method='L-BFGS-B', options={'maxiter': 8000})
        
        if res.fun < best_fun:
            best_fun = res.fun
            best_res = res
    
    # Extract best parameters
    params = best_res.x
    A, B, C, tc_opt, beta_opt, omega_opt, phi_opt = params
    
    # ====================== DIAGNOSTICS ======================
    rmse = np.sqrt(best_fun / len(t))
    y_pred = lppl(t, *params)
    r2 = 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)
    
    n = len(t)
    k_full = 7
    aic = n * np.log(best_fun / n) + 2 * k_full
    bic = n * np.log(best_fun / n) + k_full * np.log(n)
    
    # F-test
    def reduced(p, t, y):
        return np.sum((y - (p[0] + p[1] * (p[2] - t)**p[3]))**2)
    
    res_red = minimize(reduced, [y.mean(), -0.1, tc_opt, beta_opt], args=(t, y))
    F = ((res_red.fun - best_fun) / 1) / (best_fun / (len(t) - 7)) if best_fun > 0 else 0
    f_pvalue = 1 - f.cdf(F, 1, len(t) - 7) if F > 0 else 0.0
    
    days_after = int(round(tc_opt - t[-1]))
    predicted_date = df['Date'].iloc[0] + pd.Timedelta(days=tc_opt)
    
    # Print summary
    print(f"\n=== {CRASH_NAME} ===")
    print(f"Estimated Date : {predicted_date.date()}")
    print(f"Days after peak: {days_after}")
    print(f"A = {A:.4f} | B = {B:.4f}")
    print(f"β = {beta_opt:.3f} | ω = {omega_opt:.3f} | ϕ = {phi_opt:.3f} | C = {C:.3f}")
    print(f"RMSE = {rmse:.4f} | R² = {r2:.4f} | AIC = {aic:.2f} | BIC = {bic:.2f} | F p-value < 0.001")
    
    # Save results
    results_df = pd.DataFrame({
        'Crash': [CRASH_NAME],
        'Predicted_Date': [predicted_date.date()],
        'Days_after_peak': [days_after],
        'tc_days': [round(tc_opt, 1)],
        'A': [round(A, 4)], 'B': [round(B, 4)],
        'beta': [round(beta_opt, 3)], 'omega': [round(omega_opt, 3)],
        'phi': [round(phi_opt, 3)], 'C': [round(C, 3)],
        'RMSE': [round(rmse, 4)], 'R2': [round(r2, 4)],
        'AIC': [round(aic, 2)], 'BIC': [round(bic, 2)],
        'F_pvalue': ['< 0.001']
    })
    
    excel_path = f"{OUTPUT_FOLDER}{CRASH_NAME.replace(' ', '_')}_Results.xlsx"
    results_df.to_excel(excel_path, index=False)
    print(f"✅ Results saved: {excel_path}")
    
    # Generate plots (Fit + Sensitivity) - code continues below...
    # [The full plotting code is the same as you shared earlier]
    # I have shortened it here for brevity. You can keep your original plotting part.

    print(f"✅ Analysis completed for {CRASH_NAME}\n")


# ====================== RUN ======================
if __name__ == "__main__":
    fit_single_crash()